# Regresión lineal múltiple


A lo largo de los siguientes ejercicios, aprenderás a usar Python para construir un modelo de regresión lineal múltiple. Antes de comenzar con este ejercicio de programación, recomendamos encarecidamente ver la conferencia en video y completar la IVQ para los temas asociados.


Toda la información que necesitas para resolver esta tarea está en este cuaderno, y todo el código que implementarás tendrá lugar dentro de este cuaderno.


A medida que avanzamos, puedes encontrar instrucciones sobre cómo instalar las bibliotecas requeridas a medida que surjan en este cuaderno. Antes de comenzar con los ejercicios y analizar los datos, necesitamos importar todas las bibliotecas y extensiones requeridas para este ejercicio de programación. A lo largo del curso, utilizaremos pandas y statsmodels para operaciones, y seaborn para gráficos.


## Importaciones relevantes


Comience importando los paquetes y datos relevantes.


In [1]:
# Import packages
import pandas as pd
import seaborn as sns

In [2]:
# Load dataset
penguins = sns.load_dataset("penguins", cache=False)

# Examine first 5 rows of dataset
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


**Nota:** Recuerda que el valor predeterminado de `head()` es mostrar las primeras 5 filas. Si cambias el valor de `n`, puedes mostrar más filas. Por ejemplo, el comando `penguins.head(3)` mostrará 3 filas.

De las primeras 5 filas del conjunto de datos, podemos ver que hay varias columnas disponibles: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, y `sex`. También parece haber algunos datos faltantes.


## Limpieza de datos (no mostrada en el video)


Limpia el conjunto de datos seleccionando columnas específicas, renombrando columnas, eliminando filas con valores faltantes y restableciendo el índice. Para los propósitos de este ejercicio, nos centraremos en las columnas `body_mass_g`, `bill_length_mm`, `sex` y `species`. En un entorno de trabajo, deberás tomar decisiones cuidadosas sobre qué variables incluir o excluir. Más adelante en este curso, cubriremos algunas de las técnicas para la selección de variables. Por ahora, nuestro enfoque está solo en construir el modelo, y evaluar e interpretar los resultados.

**Nota:** Para los propósitos de este ejercicio, no examinamos los datos minuciosamente antes de eliminar filas con datos faltantes. En un entorno de trabajo, normalmente examinarías los datos más a fondo antes de decidir cómo manejar los datos faltantes (es decir, rellenar, eliminar, etc.). Por favor, consulta el contenido del programa anterior si necesitas revisar cómo manejar los datos faltantes.


In [3]:
# Subset data
penguins = penguins[["body_mass_g", "bill_length_mm", "sex", "species"]]

# Rename columns
penguins.columns = ["body_mass_g", "bill_length_mm", "gender", "species"]

# Drop rows with missing values
penguins.dropna(inplace=True)

# Reset index
penguins.reset_index(inplace=True, drop=True)

Puedes revisar la documentación para [`dropna()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) y [`reset_index()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html). En resumen, la función `dropna()` por defecto elimina cualquier fila con cualquier valor faltante en cualquiera de las columnas. La función `reset_index()` restablece los valores del índice para las filas en el DataFrame. Normalmente, usas `reset_index()` después de haber terminado de manipular el conjunto de datos. Al establecer `inplace=True`, no crearás un nuevo objeto DataFrame. Al establecer `drop=True`, no insertarás una nueva columna de índice en el objeto DataFrame.


In [4]:
# Examine first 5 rows of data
penguins.head()

,body_mass_g,bill_length_mm,gender,species
0,3750.0,39.1,Male,Adelie
1,3800.0,39.5,Female,Adelie
2,3250.0,40.3,Female,Adelie
3,3450.0,36.7,Female,Adelie
4,3650.0,39.3,Male,Adelie


## Crear muestra de reserva


Anteriormente, aprendiste sobre cómo crear una muestra de reserva para probar y evaluar mejor los resultados de tu modelo de regresión. Para hacer esto más fácilmente en Python, debes subconjuntar tus variables x e y, importar la función `train_test_split` de `sci-kit learn`, y luego usar la función. Por favor, revisa el contenido del curso sobre muestras de reserva según sea necesario antes de continuar con el resto del cuaderno.


In [5]:
# Subset X and y variables
penguins_X = penguins[["bill_length_mm", "gender", "species"]]
penguins_y = penguins[["body_mass_g"]]

In [6]:
# Import train-test-split function from sci-kit learn
from sklearn.model_selection import train_test_split

In [7]:
# Create training data sets and holdout (testing) data sets
X_train, X_test, y_train, y_test = train_test_split(penguins_X, penguins_y, 
                                                    test_size = 0.3, random_state = 42)

**Nota:** Hemos establecido la variable `test_size` en `0.3`, lo que indica a la función qué proporción de los datos debe estar en la muestra de reserva. Además, hemos establecido la variable `random_state` igual a `42` para reproducibilidad. Si cambias el `random_state`, tus conjuntos de datos de muestra de reserva y entrenamiento serán diferentes, por lo que tu modelo puede comportarse de manera diferente.


## Construcción del modelo


Recuerda que hemos explorado el conjunto de datos de pingüinos antes. Anteriormente, usamos gráficos de dispersión para realizar análisis exploratorios de datos, y identificamos relaciones lineales entre las siguientes variables:

* longitud del pico (mm) y longitud de la aleta (mm)
* longitud del pico (mm) y masa corporal (g)
* longitud de la aleta (mm) y masa corporal (g)

En esta parte del curso, nos centraremos en entender algunas de las relaciones de las variables con la masa corporal (g). Usaremos una variable X continua, longitud del pico (mm), y las dos variables categóricas, género y especie.

Primero, tenemos que escribir la fórmula como una cadena. Recuerda que escribimos primero el nombre de la variable y, seguido de la tilde (`~`), y luego cada una de las variables X separadas por un signo más (`+`). Podemos usar `C()` para indicar una variable categórica. Esto indicará a la función `ols()` que codifique en caliente esas variables en el modelo. Por favor, revisa los materiales del curso anterior según sea necesario para repasar cómo y por qué codificamos las variables categóricas para regresión.


In [8]:
# Write out OLS formula as a string
ols_formula = "body_mass_g ~ bill_length_mm + C(gender) + C(species)"

**Nota:** Los nombres de las variables x e y tienen que coincidir exactamente con los nombres de las columnas en el dataframe.


In [9]:
# Import ols() function from statsmodels package
from statsmodels.formula.api import ols

Después de haber importado la función `ols()`, podemos guardar `ols_data` como un dataframe, crear el objeto `ols`, ajustar el modelo y generar estadísticas resumidas. En este punto, tendría sentido verificar las suposiciones del modelo sobre los errores (homocedasticidad y normalidad de los residuos). Por favor, revisa otros recursos en el programa según sea necesario.


In [10]:
# Create OLS dataframe
ols_data = pd.concat([X_train, y_train], axis = 1)

# Create OLS object and fit the model
OLS = ols(formula = ols_formula, data = ols_data)
model = OLS.fit()

## Evaluación e interpretación del modelo


Use la función `.summary()` para obtener una tabla resumen de resultados y estadísticas del modelo.

Una vez que tenemos nuestra tabla resumen, podemos interpretar y evaluar el modelo. En la mitad superior de la tabla, obtenemos varias estadísticas resumen. Nos centraremos en `R-cuadrado`, que nos indica cuánto de la variación en la masa corporal (g) es explicada por el modelo. Un `R-cuadrado` de 0.85 es bastante alto, y esto significa que el 85% de la variación en la masa corporal (g) es explicada por el modelo.

Pasando a la mitad inferior de la tabla, obtenemos los coeficientes beta estimados por el modelo y sus intervalos de confianza del 95% y valores p correspondientes. Basándonos en la columna de valores p, etiquetada como `P>|t|`, podemos decir que todas las variables X son estadísticamente significativas, ya que el valor p es menor que 0.05 para cada variable X.


In [11]:
# Get model results
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            body_mass_g   R-squared:                       0.850
Model:                            OLS   Adj. R-squared:                  0.847
Method:                 Least Squares   F-statistic:                     322.6
Date:                Mon, 14 Nov 2022   Prob (F-statistic):           1.31e-92
Time:                        14:29:40   Log-Likelihood:                -1671.7
No. Observations:                 233   AIC:                             3353.
Df Residuals:                     228   BIC:                             3371.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===========================================================================================
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                2032.2111    354.087      5.739      0.000    1334.510    2729.913
C(gender)[T.Male]         528.9508     55.105      9.599      0.000     420.371     637.531
C(species)[T.Chinstrap]  -285.3865    106.339     -2.684      0.008    -494.920     -75.853
C(species)[T.Gentoo]     1081.6246     94.953     11.391      0.000     894.526    1268.723
bill_length_mm             35.5505      9.493      3.745      0.000      16.845      54.256
==============================================================================
Omnibus:                        0.339   Durbin-Watson:                   1.948
Prob(Omnibus):                  0.844   Jarque-Bera (JB):                0.436
Skew:                           0.084   Prob(JB):                        0.804
Kurtosis:                       2.871   Cond. No.                         798.
==============================================================================

Warnings:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Podemos entonces interpretar cada uno de los coeficientes beta para cada variable X.

### C(género) - Macho
Dado el nombre de la variable, sabemos que la variable fue codificada como `Macho = 1`, `Hembra = 0`. Esto significa que las pingüinas hembras son el punto de referencia. Si todas las demás variables son constantes, entonces esperaríamos que la masa corporal de un pingüino macho sea aproximadamente 528.95 gramos más que la masa corporal de una pingüina hembra.

### C(especie) - Chinstrap y Gentoo
Dado los nombres de estas dos variables, sabemos que los pingüinos Adelie son el punto de referencia. Entonces, si comparamos un pingüino Adelie y un pingüino Chinstrap, que tienen las mismas características excepto su especie, esperaríamos que el pingüino Chinstrap tenga una masa corporal de aproximadamente 285.39 gramos menos que el pingüino Adelie. Si comparamos un pingüino Adelie y un pingüino Gentoo, que tienen las mismas características excepto su especie, esperaríamos que el pingüino Gentoo tenga una masa corporal de aproximadamente 1,081.62 gramos más que el pingüino Adelie.

### Longitud del pico (mm)
Por último, la longitud del pico (mm) es una variable continua, por lo que si comparamos dos pingüinos que tienen las mismas características, excepto que el pico de un pingüino es 1 milímetro más largo, esperaríamos que el pingüino con el pico más largo tenga 35.55 gramos más de masa corporal que el pingüino con el pico más corto.



**¡Felicidades!** Has completado este laboratorio. Sin embargo, es posible que no notes una marca de verificación verde junto a este elemento en la plataforma de Coursera. Por favor, continúa con tu progreso independientemente de la marca de verificación. Solo haz clic en el icono de "guardar" en la parte superior de este cuaderno para asegurarte de que tu trabajo ha sido registrado.

Ahora entiendes cómo construir un modelo de regresión lineal múltiple con Python. En adelante, puedes comenzar a usar modelos de regresión lineal múltiple con tus propios conjuntos de datos.

